<a href="https://colab.research.google.com/github/shabir-mp/AIrena-Comptetition-Project/blob/main/trial1_penyisihanAIrena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# - Baca train/test
# - Parsing waktu ke detik
# - Feature engineering sederhana
# - Stratified K-Fold + LightGBM (fallback RandomForest)
# - Hitung OOF F1-macro dan buat submission.csv

import os, re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from lightgbm import early_stopping, log_evaluation

# optional: try import lightgbm, otherwise fallback to RandomForest
use_lgb = True
try:
    import lightgbm as lgb
except Exception:
    use_lgb = False
    from sklearn.ensemble import RandomForestClassifier

# ---------- Settings ----------
TRAIN_PATH = "train.csv"   # letakkan file di folder yang sama
TEST_PATH  = "test.csv"
OUT_SUB   = "submission.csv"

# ---------- Utility: parse time strings ----------
def parse_time_to_seconds(x):
    """
    Convert strings like:
      - '8d 23:58:12.145' -> seconds
      - '1:07:56.830'     -> seconds
    Return float seconds or np.nan
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.match(r'(?:(\\d+)d\\s+)?(\\d{1,2}):(\\d{2}):([\\d\\.]+)', s)
    if m:
        days = int(m.group(1)) if m.group(1) else 0
        hh = int(m.group(2))
        mm = int(m.group(3))
        ss = float(m.group(4))
        return days*86400 + hh*3600 + mm*60 + ss
    try:
        return float(s)
    except:
        return np.nan

def find_time_cols(df):
    return [c for c in df.columns if ('waktu' in c.lower()) or ('durasi' in c.lower()) or ('time' in c.lower())]

# ---------- Load ----------
assert os.path.exists(TRAIN_PATH), f"{TRAIN_PATH} not found"
assert os.path.exists(TEST_PATH), f"{TEST_PATH} not found"

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print("Loaded:", train.shape, test.shape)

# ---------- Detect time-like columns and parse ----------
time_cols = sorted(list(set(find_time_cols(train) + find_time_cols(test))))
print("Detected time columns:", time_cols)
for c in time_cols:
    newc = c + "_s"
    if c in train.columns:
        train[newc] = train[c].apply(parse_time_to_seconds)
    if c in test.columns:
        test[newc]  = test[c].apply(parse_time_to_seconds)

# ---------- Feature engineering function ----------
def add_basic_features(df):
    df = df.copy()
    # Simulation summary
    sim_cols = [c for c in df.columns if 'waktu_simulasi' in c and c.endswith('_s')]
    df['sim_time_count'] = df[sim_cols].notnull().sum(axis=1) if sim_cols else 0
    if sim_cols:
        df['sim_time_min'] = df[sim_cols].min(axis=1)
        df['sim_time_mean'] = df[sim_cols].mean(axis=1)
        df['sim_time_median'] = df[sim_cols].median(axis=1)
    else:
        df['sim_time_min'] = df['sim_time_mean'] = df['sim_time_median'] = np.nan

    # uji_lingkungan & kalibrasi flags
    for letter in ['A','B','C']:
        col = f"waktu_uji_lingkungan_{letter}_s"
        df[f'has_uji_{letter}'] = df[col].notnull().astype(int) if col in df.columns else 0
        colk = f"waktu_kalibrasi_lapangan_{letter}_s"
        df[f'has_kalib_{letter}'] = df[colk].notnull().astype(int) if colk in df.columns else 0

    # demo flag
    df['has_demo'] = df['waktu_demonstrasi_lapangan_s'].notnull().astype(int) if 'waktu_demonstrasi_lapangan_s' in df.columns else 0

    # coordinates magnitude
    if all(col in df.columns for col in ('koordinat_X_lapangan','koordinat_Y_lapangan','koordinat_Z_lapangan')):
        df['coord_mag'] = ((df['koordinat_X_lapangan'].fillna(0).astype(float)**2) +
                           (df['koordinat_Y_lapangan'].fillna(0).astype(float)**2) +
                           (df['koordinat_Z_lapangan'].fillna(0).astype(float)**2))**0.5
    else:
        df['coord_mag'] = 0.0

    # speed proxy (distance / mean sim time)
    if 'total_jarak_tempuh_satu_siklus' in df.columns:
        df['total_jarak_tempuh_satu_siklus'] = pd.to_numeric(df['total_jarak_tempuh_satu_siklus'], errors='coerce')
        df['speed_sim_mean'] = df['total_jarak_tempuh_satu_siklus'] / (df['sim_time_mean'].replace(0, np.nan) + 1e-6)
    else:
        df['speed_sim_mean'] = np.nan

    return df

train_fe = add_basic_features(train)
test_fe = add_basic_features(test)

# ---------- Frequency encoding (simple) ----------
freq_cols = [c for c in ['tim_penjelajah','tim_mesin','nama_penjelajah','planet_tim_mesin'] if c in train_fe.columns]
for c in freq_cols:
    freq = train_fe[c].fillna("NA").value_counts().to_dict()
    train_fe[c + "_freq"] = train_fe[c].fillna("NA").map(freq).astype(float)
    if c in test_fe.columns:
        test_fe[c + "_freq"] = test_fe[c].fillna("NA").map(freq).fillna(0).astype(float)

# ---------- Build feature list ----------
possible_numeric = ['banyak_siklus','total_jarak_tempuh','total_jarak_tempuh_satu_siklus','banyak_titik_navigasi',
                    'radius_planet_penjelajahan','umur_biometrik_penjelajah','total_siklus_uji','siklus_kalibrasi_lapangan',
                    'siklus_simulasi_alpha','siklus_simulasi_beta','siklus_simulasi_gamma','siklus_demonstrasi_lapangan']
numeric_keep = [c for c in possible_numeric if c in train_fe.columns]

engineered = [c for c in train_fe.columns if any(s in c for s in ['_s','_count','_min','_mean','_median','_freq','_mag','speed_sim_mean'])]
numeric_keep += engineered

# remove train-only leakage columns if they exist
leak_cols = ['durasi_misi_diselesaikan','jumlah_isi_bensin_selama_misi','peringkat_durasi_misi_tercepat','penghargaan_misi']
numeric_keep = [c for c in set(numeric_keep) if c not in leak_cols]
numeric_keep = sorted([c for c in numeric_keep if c in train_fe.columns])

# label encode some categorical columns (combined train+test)
cat_cols = [c for c in ['wilayah_eksplorasi','jenis_medan','orientasi_rute','planet_eksplorasi','sektor_planet_eksplorasi',
                        'planet_lahir','planet_kewarganegaraan','planet_tim_penjelajah','sektor_tim_penjelajah',
                        'planet_tim_mesin','sektor_tim_mesin','tim_penjelajah','tim_mesin'] if c in train_fe.columns]

for c in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_fe[c].astype(str), test_fe[c].astype(str)], axis=0).fillna("NA")
    le.fit(combined)
    train_fe[c + "_le"] = le.transform(train_fe[c].astype(str).fillna("NA"))
    test_fe[c + "_le"]  = le.transform(test_fe[c].astype(str).fillna("NA"))

le_cols = [c + "_le" for c in cat_cols]
freq_cols_added = [c + "_freq" for c in freq_cols if (c + "_freq") in train_fe.columns]


final_features = sorted(list(set(numeric_keep + le_cols + freq_cols_added)))
final_features = [f for f in final_features if f != 'id']

# --- FIX: remove features not available in test ---
final_features = [f for f in final_features if f in train_fe.columns and f in test_fe.columns]

# Filter out the original time columns which are of 'object' dtype.
# These columns are problematic for LightGBM, and their numeric '_s' counterparts are already included.
final_features = [f for f in final_features if f not in time_cols]

print("Final usable features:", len(final_features))

# ---------- Prepare X, y ----------
X = train_fe[final_features].copy()
X_test = test_fe[final_features].copy()
y = train_fe['penghargaan_misi'].copy()

target_le = LabelEncoder()
y_enc = target_le.fit_transform(y)
print("Target classes:", list(target_le.classes_))

# ---------- CV training ----------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(X), len(target_le.classes_)))
test_probs = np.zeros((len(X_test), len(target_le.classes_)))

if use_lgb:
    lgb_params = {
        'objective':'multiclass',
        'num_class': len(target_le.classes_),
        'learning_rate':0.05,
        'n_estimators':2000,
        'verbosity':-1,
        'random_state':42,
        'num_threads':2
    }
else:
    rf_params = {'n_estimators':300, 'random_state':42, 'n_jobs':-1}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_enc), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_enc[tr_idx], y_enc[val_idx]
    if use_lgb:
        model = lgb.LGBMClassifier(**lgb_params)
        # Perubahan 1
        model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        early_stopping(100),
        log_evaluation(0)   # 0 = silent
    ]
)
        val_proba = model.predict_proba(X_val)
        test_p = model.predict_proba(X_test)
    else:
        model = RandomForestClassifier(**rf_params)
        model.fit(X_tr, y_tr)
        val_proba = model.predict_proba(X_val)
        test_p = model.predict_proba(X_test)
    oof_probs[val_idx] = val_proba
    test_probs += test_p / skf.n_splits
    val_pred = val_proba.argmax(axis=1)
    print(f"Fold {fold} F1-macro:", round(f1_score(y_val, val_pred, average='macro'), 4))

oof_pred = oof_probs.argmax(axis=1)
print("OOF F1-macro overall:", round(f1_score(y_enc, oof_pred, average='macro'), 4))

# ---------- Prepare submission ----------
test_pred_idx = test_probs.argmax(axis=1)
test_pred_labels = target_le.inverse_transform(test_pred_idx)
submission = pd.DataFrame({
    'id': test_fe['id'] if 'id' in test_fe.columns else np.arange(1, len(test_pred_labels)+1),
    'penghargaan_misi': test_pred_labels
})
submission.to_csv(OUT_SUB, index=False)
print("Saved submission to:", OUT_SUB)
print(submission.head(10))


Loaded: (5508, 52) (1378, 48)
Detected time columns: ['durasi_misi_diselesaikan', 'peringkat_durasi_misi_tercepat', 'waktu_demonstrasi_lapangan', 'waktu_kalibrasi_lapangan_A', 'waktu_kalibrasi_lapangan_B', 'waktu_kalibrasi_lapangan_C', 'waktu_simulasi_alpha', 'waktu_simulasi_beta', 'waktu_simulasi_gamma', 'waktu_uji_lingkungan_A', 'waktu_uji_lingkungan_B', 'waktu_uji_lingkungan_C']
Final usable features: 45
Target classes: ['Bronze', 'Gold', 'Merit', 'Mission Incomplete', 'Silver']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[72]	valid_0's multi_logloss: 0.676744
Fold 1 F1-macro: 0.6567
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[106]	valid_0's multi_logloss: 0.625775
Fold 2 F1-macro: 0.6551
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[91]	valid_0's multi_logloss: 0.667645
Fold 3 F1-macro: 0.6539
Training until validation scores